In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [4]:
louisiana

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns
0,0,louisiana,0,0,367983.8318,68239.85746,80.234375,79198.54166,6714.566582,1154589.900,...,9.980000,0.322352,2.821843,4.032755,0.0,39.360000,15.368289,106.770000,4.290000,32.920000
1,0,louisiana,1,0,362385.6368,67201.71394,79.013757,77993.68198,6612.416842,1137024.945,...,9.968923,0.319496,2.800291,4.092947,0.0,40.166298,14.997723,108.605133,3.731796,31.374749
2,0,louisiana,2,0,361096.1267,66962.58391,78.732596,77716.14990,6588.887271,1132978.965,...,9.489254,0.320059,2.788854,4.099808,0.0,38.639666,13.763035,80.719306,3.559992,31.843131
3,0,louisiana,3,0,359935.7476,66747.40026,78.479589,77466.40976,6567.713942,1129338.147,...,10.279229,0.321689,2.777719,4.162110,0.0,41.465412,14.851001,146.300301,3.841408,33.650089
4,0,louisiana,4,0,358773.5212,66531.87404,78.226180,77216.27204,6546.506906,1125691.534,...,9.983308,0.323218,2.772568,4.173150,0.0,37.807488,13.488061,146.740554,3.879013,34.227053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,127127,louisiana,31,0,311979.6878,53280.76885,83.099337,74341.21831,6871.576034,1116082.754,...,2.290044,0.099194,1.610628,2.566150,0.0,9.504129,1.357871,6.354095,0.494434,17.846785
2408,127127,louisiana,32,0,310670.5663,53049.87938,83.075690,74040.77745,6869.511396,1114713.368,...,2.285685,0.087899,1.527726,2.529939,0.0,9.783970,1.280190,4.725632,0.364936,17.059667
2409,127127,louisiana,33,0,309356.4650,52821.26235,83.037733,73734.23638,6866.309713,1113193.917,...,2.281655,0.076939,1.445208,2.497102,0.0,9.535228,1.030977,3.223424,0.239272,16.254621
2410,127127,louisiana,34,0,308044.9743,52594.98521,82.990728,73425.33726,6862.387927,1111581.984,...,2.277804,0.066352,1.363483,2.466951,0.0,9.696216,0.918047,1.841394,0.117569,15.427843


In [5]:
# 1) Filter out the base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()


In [6]:
# 2) Define your fuels and sectors
relevant_gases = ['n2o',
                  'sf6',
                  'c4f6',
                  'c4f8o',
                  'c2f6',
                  'c3f8',
                  'c6f14',
                  'c5f8',
                  'cc4f8']

sectors = ['chemicals',
           'electronics',
           'metals']



In [13]:
# 3) Initialize accumulators
emissions_avoided_by_sector = pd.DataFrame({'time_period': base_case['time_period']}, index=base_case.index)

In [8]:
# 4) Industrial cost parameters
capex_chemicals_n2o = 20*0.2
opex_chemicals_n2o = 20*0.8
capex_electronics_sf6 = 40*0.5
opex_electronics_sf6 = 40*0.5
capex_electronics_c4f6 = 40*0.5
opex_electronics_c4f6 = 40*0.5
capex_electronics_c2f6 = 40*0.5
opex_electronics_c2f6 = 40*0.5
capex_electronics_c3f8 = 40*0.5
opex_electronics_c3f8 = 40*0.5
capex_electronics_c5f8 = 40*0.5
opex_electronics_c5f8 = 40*0.5
capex_electronics_cc4f8 = 40*0.5
opex_electronics_cc4f8 = 40*0.5
capex_metals_sf6 = 20*0.7
opex_metals_sf6 = 20*0.7
capex_metals_c2f6 = 20*0.7
opex_metals_c2f6 = 20*0.7

In [14]:
# 5) Loop over fuels and sectors
for gas in relevant_gases:
    for sector in sectors:
        abatement_cols = [c for c in base_case.columns
                if c.startswith(f'ef_ippu_tonne_{gas}_per_tonne_production_{sector}')]
   
        emissions_cols = [c for c in base_case.columns
                if c.startswith(f'emission_co2e_{gas}_ippu_production_{sector}')]

        if(len(abatement_cols)>0 and len(emissions_cols)>0):
            abatement_factor = base_case[abatement_cols[0]]
            emissions = base_case[emissions_cols[0]]

            if(abatement_factor.sum()>0):
                abatement_factor = abatement_factor[0]/abatement_factor
                emissions_avoided_by_sector[f'emissions_abated_{sector}_{gas}'] = (abatement_factor-1)*emissions
                emissions_avoided_by_sector[f'capex_emissions_abated_{sector}_{gas}'] = (abatement_factor-1)*emissions*globals()[f'capex_{sector}_{gas}']
                emissions_avoided_by_sector[f'opex_emissions_abated_{sector}_{gas}'] = (abatement_factor-1)*emissions*globals()[f'opex_{sector}_{gas}']



    


In [15]:
#ccs
emissions_avoided_by_sector['ccs'] = base_case['emission_co2e_subsector_total_ccsq'] - base_case['emission_co2e_subsector_total_ccsq'][0]
capex_ccs = 500*(0.96)^emissions_avoided_by_sector['time_period']
opex_ccs = 50*(0.96)^emissions_avoided_by_sector['time_period']
emissions_avoided_by_sector['capex_ccs']<-emissions_avoided_by_sector['ccs']*capex_ccs
emissions_avoided_by_sector['opex_ccs']<-emissions_avoided_by_sector['ccs']*opex_ccs


In [18]:


# 6) Write out to CSV

emissions_avoided_by_sector.to_csv(OUTPUT_DIR/'fugitive_emissions_and_ccs.csv', index=False)
